# CodeNova Task 2: Data Cleaning Report

This notebook audits the supplied sales dataset, demonstrates reproducible cleaning, and reviews before/after quality metrics. The raw dataset remains unchanged. Outliers are flagged for review rather than automatically removed.

## 1. Imports and paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RAW_PATH = ROOT / "data" / "raw" / "original_dataset.csv"
print("Project root:", ROOT)
print("Raw data exists:", RAW_PATH.exists())

Project root: /mnt/data/CodeNova_Task2_Data_Cleaning
Raw data exists: True


## 2. Load and inspect the original data

In [2]:
raw = pd.read_csv(RAW_PATH)
print("Shape:", raw.shape)
display(raw.head())
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "unique_values": raw.nunique(dropna=True)}))

Shape: (360, 12)


,Order_ID,Order_Date,Product,Category,Region,Sales_Channel,Customer_Type,Quantity,Unit_Price_INR,Unit_Cost_INR,Discount_Percent,Payment_Method
0,ORD0329,2025-01-02,Water Bottle,Lifestyle,Mumbai,Online,Returning,1,650.0,274.4,0.00,Cash
1,ORD0084,2025-01-03,Office Chair,Furniture,Bengaluru,Retail Store,Returning,2,8925.0,6000.0,0.00,Cash
2,ORD0233,2025-01-04,Smartphone,Electronics,Pune,Online,New,5,18900.0,13770.0,0.15,Cash
3,ORD0110,2025-01-09,Pen Set,Stationery,Bengaluru,Distributor,Returning,4,209.0,91.8,0.05,Credit Card
4,ORD0257,2025-01-10,Backpack,Lifestyle,Mumbai,Online,New,1,1680.0,867.0,0.05,Credit Card


,dtype,missing,unique_values
Order_ID,object,0,360
Order_Date,object,0,266
Product,object,0,12
Category,object,0,5
Region,object,0,6
Sales_Channel,object,0,3
Customer_Type,object,0,2
Quantity,int64,0,5
Unit_Price_INR,float64,0,36
Unit_Cost_INR,float64,0,36


## 3. Missing values, duplicates, and structural checks

In [3]:
quality_checks = pd.DataFrame({
    "check": ["Missing cells", "Exact duplicate rows", "Duplicate Order_ID values", "Blank text cells", "Unparseable dates"],
    "count": [int(raw.isna().sum().sum()), int(raw.duplicated().sum()), int(raw["Order_ID"].duplicated().sum()),
              int(sum(raw[c].astype("string").str.strip().eq("").fillna(False).sum() for c in raw.select_dtypes(include="object").columns)),
              int(pd.to_datetime(raw["Order_Date"], errors="coerce").isna().sum())]
})
display(quality_checks)

,check,count
0,Missing cells,0
1,Exact duplicate rows,0
2,Duplicate Order_ID values,0
3,Blank text cells,0
4,Unparseable dates,0


## 4. Logical validation and numeric outliers

In [4]:
numeric_cols = ["Quantity", "Unit_Price_INR", "Unit_Cost_INR", "Discount_Percent"]
logical_checks = pd.DataFrame({
    "check": ["Quantity <= 0", "Unit price <= 0", "Unit cost < 0", "Discount outside [0, 1]", "Unit cost > unit price"],
    "count": [int((raw["Quantity"] <= 0).sum()), int((raw["Unit_Price_INR"] <= 0).sum()), int((raw["Unit_Cost_INR"] < 0).sum()),
              int(((raw["Discount_Percent"] < 0) | (raw["Discount_Percent"] > 1)).sum()), int((raw["Unit_Cost_INR"] > raw["Unit_Price_INR"]).sum())]
})
display(logical_checks)

outlier_rows = []
for col in numeric_cols:
    s = pd.to_numeric(raw[col], errors="coerce").dropna()
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({"column": col, "lower_fence": lo, "upper_fence": hi, "outlier_count": int(((s < lo) | (s > hi)).sum())})
display(pd.DataFrame(outlier_rows))

,check,count
0,Quantity <= 0,0
1,Unit price <= 0,0
2,Unit cost < 0,0
3,"Discount outside [0, 1]",0
4,Unit cost > unit price,0


,column,lower_fence,upper_fence,outlier_count
0,Quantity,-2.00,6.00,0
1,Unit_Price_INR,-24150.00,43290.00,35
2,Unit_Cost_INR,-18780.00,32868.00,35
3,Discount_Percent,-0.15,0.25,0


## 5. Run the reproducible cleaning script

In [5]:
import subprocess, sys
result = subprocess.run([sys.executable, str(ROOT / "src" / "data_cleaning.py")], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Cleaning script failed")

DATA CLEANING COMPLETED
{
  "before": {
    "rows": 360,
    "columns": 12,
    "total_missing_cells": 0,
    "columns_with_missing_values": 0,
    "exact_duplicate_rows": 0,
    "duplicate_order_ids": 0,
    "invalid_dates": 0,
    "blank_string_cells": 0
  },
  "after": {
    "rows": 360,
    "columns": 12,
    "total_missing_cells": 0,
    "columns_with_missing_values": 0,
    "exact_duplicate_rows": 0,
    "duplicate_order_ids": 0,
    "invalid_dates": 0,
    "blank_string_cells": 0
  },
  "logical_validation_checks": {
    "quantity_nonpositive": 0,
    "unit_price_nonpositive": 0,
    "unit_cost_negative": 0,
    "discount_out_of_range": 0,
    "cost_greater_than_price": 0,
    "duplicate_order_ids": 0
  }
}
Clean dataset: /mnt/data/CodeNova_Task2_Data_Cleaning/data/cleaned/cleaned_dataset.csv



## 6. Before-and-after comparison

In [6]:
comparison = pd.read_csv(ROOT / "reports" / "before_after_comparison.csv")
display(comparison)
numeric_profile = pd.read_csv(ROOT / "reports" / "numeric_profile_before_after.csv")
display(numeric_profile)

,metric,before,after
0,rows,360,360
1,columns,12,12
2,total_missing_cells,0,0
3,columns_with_missing_values,0,0
4,exact_duplicate_rows,0,0
5,duplicate_order_ids,0,0
6,invalid_dates,0,0
7,blank_string_cells,0,0


,stage,column,count,missing,mean,median,std,min,q1,q3,max
0,before,Quantity,360,0,2.108333,2.00,1.256487,1.0,1.0,3.0,5.00
1,after,Quantity,360,0,2.108333,2.00,1.256487,1.0,1.0,3.0,5.00
2,before,Unit_Price_INR,360,0,11806.720833,2500.00,15575.018903,209.0,1140.0,18000.0,54600.00
3,after,Unit_Price_INR,360,0,11806.720833,2500.00,15575.018903,209.0,1140.0,18000.0,54600.00
4,before,Unit_Cost_INR,360,0,8982.822222,1372.00,12547.391177,88.2,588.0,13500.0,42840.00
5,after,Unit_Cost_INR,360,0,8982.822222,1372.00,12547.391177,88.2,588.0,13500.0,42840.00
6,before,Discount_Percent,360,0,0.063333,0.05,0.058200,0.0,0.0,0.1,0.15
7,after,Discount_Percent,360,0,0.063333,0.05,0.058200,0.0,0.0,0.1,0.15


## 7. Cleaning decisions and conclusion

The source file is preserved. Since this dataset has no observed missing cells, exact duplicate rows, duplicate order IDs, invalid dates, blank strings, or basic logical violations, forced imputation or record removal would not be justified. IQR-flagged price/cost values should be reviewed in product context because genuine differences among product price levels can create statistical outliers. The generated Markdown audit report documents the observed findings and all output files.